<a href="https://colab.research.google.com/github/omerlutfiduran/senior-design-project/blob/main/Senior_Desing_Project_Data_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import ee
ee.Authenticate()

True

In [42]:
ee.Initialize(project='senior-design-project-489117')


In [43]:
import ee
import folium

# Antalya sınırlarını tanımlama (FAO GAUL Level 1 veri setini kullanıyoruz)
antalya_fc = ee.FeatureCollection("FAO/GAUL/2015/level1") \
               .filter(ee.Filter.eq('ADM1_NAME', 'Antalya'))

# NASA FIRMS Aktif Yangın Veri Setini Çağırma
# Test için büyük Manavgat yangınlarının olduğu 2021 tarihini seçiyoruz.
start_date = '2021-07-01'
end_date = '2021-08-31'

firms_dataset = ee.ImageCollection('FIRMS') \
                  .filterBounds(antalya_fc) \
                  .filterDate(start_date, end_date)

# Sadece ısı anormalliği olan pikselleri (T21 bandı) seçip Antalya sınırına kırpıyoruz
fires = firms_dataset.select('T21').max().clip(antalya_fc)

# Harita Üzerinde Görselleştirme
# GEE görüntülerini Folium haritasına eklemek için yardımcı fonksiyon
def add_ee_layer(self, ee_image_object, vis_params, name):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

# Antalya merkezli interaktif bir harita oluşturalım
my_map = folium.Map(location=[36.9, 30.7], zoom_start=8)

# Antalya sınırını haritaya ekleme
empty = ee.Image().byte()
outline = empty.paint(featureCollection=antalya_fc, color=1, width=2)
my_map.add_ee_layer(outline, {'palette': ['000000']}, 'Antalya Siniri')

# Yangın noktalarını haritaya ekleme (Isı şiddetine göre renk gradyanı)
# 300 Kelvin (sarı) ile 500 Kelvin (koyu kırmızı) arasında doğrusal bir renk geçişi sağlanır.
fire_vis_params = {
    'min': 300,
    'max': 500,
    'palette': ['yellow', 'orange', 'red', 'darkred']
}
my_map.add_ee_layer(fires, fire_vis_params, '2021 Yanginlari (FIRMS - Isi Siddeti)')

# Harita katman kontrolünü ekle ve göster
my_map.add_child(folium.LayerControl())
display(my_map)

In [44]:
import pandas as pd
import ee

# 1. Yangın Olan Noktaları (1) Çekme
# Haritadaki kırmızı piksellerden test amaçlı 200 tanesinin koordinatını alıyoruz
fire_samples = fires.sample(
    region=antalya_fc,
    scale=1000,       # MODIS uydusunun çözünürlüğü (1km)
    numPixels=200,    # Test aşamasında olduğumuz için şimdilik 200 satır
    geometries=True
)

def add_fire_label(feature):
    return feature.set('YANGIN_DURUMU', 1)

fire_labeled = fire_samples.map(add_fire_label)

# 2. Yangın Olmayan Noktaları (0) Oluşturma
# Antalya sınırları içine rastgele 200 nokta atıyoruz (Sağlıklı orman / Çoğunluk Sınıfı)
non_fire_samples = ee.FeatureCollection.randomPoints(region=antalya_fc, points=200)

def add_non_fire_label(feature):
    return feature.set('YANGIN_DURUMU', 0)

non_fire_labeled = non_fire_samples.map(add_non_fire_label)

# 3. İki Veri Setini Birleştirme (Toplam 400 satırlık ham verimiz)
combined_points = fire_labeled.merge(non_fire_labeled)

# 4. GEE Verisini Python Tablosuna (Pandas DataFrame) Çevirme
# Sistemden Enlem, Boylam ve Etiket (0/1) bilgilerini sıyırıyoruz
def extract_coords_and_labels(feature):
    coords = feature.geometry().coordinates()
    label = feature.get('YANGIN_DURUMU')
    return ee.Feature(None, {'Boylam': coords.get(0), 'Enlem': coords.get(1), 'YANGIN_DURUMU': label})

final_dataset_ee = combined_points.map(extract_coords_and_labels)

# Veriyi Google sunucularından bizim Colab ortamımıza indiriyoruz
data_list = final_dataset_ee.reduceColumns(ee.Reducer.toList(3), ['Enlem', 'Boylam', 'YANGIN_DURUMU']).get('list').getInfo()

# Python (Pandas) tablosu oluşturma
df = pd.DataFrame(data_list, columns=['Enlem', 'Boylam', 'YANGIN_DURUMU'])

# Tablonun ilk 5 ve son 5 satırını gözlem için ekrana bastırma
print("\nTabo oluşturuldu")
display(df.head())
display(df.tail())


Tabo oluşturuldu


,Enlem,Boylam,YANGIN_DURUMU
0,36.999414,31.367858,1
1,36.884091,31.390846,1
2,36.992086,31.340318,1
3,36.917897,31.051564,1
4,36.843344,31.537043,1


,Enlem,Boylam,YANGIN_DURUMU
206,36.977152,29.925556,0
207,36.732105,29.726226,0
208,37.296497,30.324046,0
209,37.206580,31.508505,0
210,36.903078,29.943865,0


In [45]:
import pandas as pd
import ee

#ÖNCEKİ BAŞARILI ADIMLAR (Topografya ve Meteoroloji)
elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Yukseklik')
slope_image = ee.Terrain.slope(elevation_image).rename('Egim')

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
         .filterDate('2021-07-01', '2021-08-31') \
         .mean()
temp_c_image = era5.select('temperature_2m').subtract(273.15).rename('Sicaklik_C')

u_wind = era5.select('u_component_of_wind_10m')
v_wind = era5.select('v_component_of_wind_10m')
wind_speed_image = u_wind.pow(2).add(v_wind.pow(2)).sqrt().rename('Ruzgar_Hizi')

# BİTKİ ÖRTÜSÜ / YAKIT (MODIS NDVI)
# MOD13A2 veri setinden 2021 yaz dönemi ortalamasını çekip 0.0001 Scale Factor uyguluyoruz
ndvi_image = ee.ImageCollection("MODIS/061/MOD13A2") \
               .filterDate('2021-07-01', '2021-08-31') \
               .select('NDVI') \
               .mean() \
               .multiply(0.0001).rename('NDVI')

#Tüm Verileri Tek Haritada Birleştirme (Yükseklik + Eğim + Sıcaklık + Rüzgar + NDVI)
combined_image = ee.Image.cat([
    elevation_image, slope_image, temp_c_image, wind_speed_image, ndvi_image
])

#Koordinatlara Verileri Zımbalama
points_with_data = combined_image.sampleRegions(
    collection=combined_points,
    scale=1000,
    geometries=True
)

#Python Tablosuna Çevirme İşlemi
def extract_all_features(feature):
    coords = feature.geometry().coordinates()
    return ee.Feature(None, {
        'Boylam': coords.get(0),
        'Enlem': coords.get(1),
        'Yukseklik': feature.get('Yukseklik'),
        'Egim': feature.get('Egim'),
        'Sicaklik_C': feature.get('Sicaklik_C'),
        'Ruzgar_Hizi': feature.get('Ruzgar_Hizi'),
        'NDVI': feature.get('NDVI'), # Yeni eklenen NDVI sütunu
        'YANGIN_DURUMU': feature.get('YANGIN_DURUMU')
    })

final_dataset_step7 = points_with_data.map(extract_all_features)

# Veriyi Çekip Pandas Tablosu Yapma
columns_to_fetch = ['Enlem', 'Boylam', 'Yukseklik', 'Egim', 'Sicaklik_C', 'Ruzgar_Hizi', 'NDVI', 'YANGIN_DURUMU']

data_list_step7 = final_dataset_step7.reduceColumns(
    ee.Reducer.toList(len(columns_to_fetch)), columns_to_fetch
).get('list').getInfo()

# Eksik verileri temizle
df_final = pd.DataFrame(data_list_step7, columns=columns_to_fetch).dropna()


display(df_final.head())
display(df_final.tail())

,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,Ruzgar_Hizi,NDVI,YANGIN_DURUMU
0,36.998776,31.368956,302,6.577698,27.957246,0.691825,0.375050,1
1,36.881995,31.386922,134,3.956093,29.786247,0.440963,0.396200,1
2,36.989793,31.342006,333,2.475168,29.828557,0.631214,0.390475,1
3,36.917928,31.054546,11,0.147884,30.978833,0.607256,0.441300,1
4,36.846062,31.539636,67,2.416106,29.078005,0.360979,0.292475,1


,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,Ruzgar_Hizi,NDVI,YANGIN_DURUMU
198,36.980810,29.922668,1476,0.446865,21.663661,0.892307,0.240950,0
199,36.729281,29.725039,1670,10.803011,22.546458,0.480758,0.225200,0
200,37.295220,30.326910,823,0.093112,25.185609,1.007540,0.243800,0
201,37.205389,31.512686,1843,10.389173,21.070538,1.141420,0.327375,0
202,36.899961,29.940635,1786,8.519497,22.301253,0.774359,0.229325,0


In [46]:
import pandas as pd
import ee

# Tablomuzdaki 200'er koordinat için Yükseklik Verisini SRTM uydusundan alma
# Tüm dünya için yükseklik modelini alıyoruz
elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Yukseklik')

# Koordinatlara Yükseklik Değerini Zımbalama
# Elimizdeki 400 noktalı (combined_points) veriyi bu haritanın üzerine atıp
# o noktalara denk gelen yükseklik değerlerini alıyoruz.
points_with_elevation = elevation_image.sampleRegions(
    collection=combined_points, # Bir önceki aşamada oluşturduğumuz 0 ve 1'ler
    scale=1000,                 # 1 km çözünürlük
    geometries=True
)

#Python Tablosuna Çevirme İşlemi
def extract_all_features(feature):
    coords = feature.geometry().coordinates()
    label = feature.get('YANGIN_DURUMU')
    elev = feature.get('Yukseklik')
    return ee.Feature(None, {
        'Boylam': coords.get(0),
        'Enlem': coords.get(1),
        'Yukseklik': elev,
        'YANGIN_DURUMU': label
    })

final_dataset_step4 = points_with_elevation.map(extract_all_features)

#Veriyi çekip liste yapma
data_list_step4 = final_dataset_step4.reduceColumns(
    ee.Reducer.toList(4), ['Enlem', 'Boylam', 'Yukseklik', 'YANGIN_DURUMU']
).get('list').getInfo()

#Pandas Tablosuna dönüştürme
df_elev = pd.DataFrame(data_list_step4, columns=['Enlem', 'Boylam', 'Yukseklik', 'YANGIN_DURUMU'])

# Sonuçlar
display(df_elev.head())
display(df_elev.tail())

,Enlem,Boylam,Yukseklik,YANGIN_DURUMU
0,36.998776,31.368956,302,1
1,36.881995,31.386922,134,1
2,36.989793,31.342006,333,1
3,36.917928,31.054546,11,1
4,36.846062,31.539636,67,1


,Enlem,Boylam,Yukseklik,YANGIN_DURUMU
206,36.980810,29.922668,1476,0
207,36.729281,29.725039,1670,0
208,37.295220,30.326910,823,0
209,37.205389,31.512686,1843,0
210,36.899961,29.940635,1786,0


In [47]:
import pandas as pd
import ee

#Yükseklik Verisini (SRTM Uydusu) Çağırma
elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Yukseklik')

#Yükseklik verisini kullanarak Eğim (Slope) haritasını oluşturma
slope_image = ee.Terrain.slope(elevation_image).rename('Egim')

#Yükseklik ve Eğimi tek bir haritada (iki bantlı görüntü) birleştirme
#Böylece iğneyi batırdığımızda iki veriyi aynı anda çekeceğiz
topo_image = ee.Image.cat([elevation_image, slope_image])

#Koordinatlara Yükseklik ve Eğim Değerlerini Zımbalama
points_with_topo = topo_image.sampleRegions(
    collection=combined_points, # Bir önceki aşamada oluşturduğumuz 0 ve 1'ler
    scale=1000,                 # 1 km çözünürlük
    geometries=True
)

# Python Tablosuna Çevirme İşlemi
def extract_all_features(feature):
    coords = feature.geometry().coordinates()
    label = feature.get('YANGIN_DURUMU')
    elev = feature.get('Yukseklik')
    slope_val = feature.get('Egim') # Yeni eklenen sütun

    return ee.Feature(None, {
        'Boylam': coords.get(0),
        'Enlem': coords.get(1),
        'Yukseklik': elev,
        'Egim': slope_val,       # Yeni eklenen sütun
        'YANGIN_DURUMU': label
    })

final_dataset_step4 = points_with_topo.map(extract_all_features)

# Veriyi çekip liste yapma
data_list_step4 = final_dataset_step4.reduceColumns(
    ee.Reducer.toList(5), ['Enlem', 'Boylam', 'Yukseklik', 'Egim', 'YANGIN_DURUMU']
).get('list').getInfo()

# Pandas Tablosuna dönüştürme
df_topo = pd.DataFrame(data_list_step4, columns=['Enlem', 'Boylam', 'Yukseklik', 'Egim', 'YANGIN_DURUMU'])

# Sonuçlar

display(df_topo.head())
display(df_topo.tail())

,Enlem,Boylam,Yukseklik,Egim,YANGIN_DURUMU
0,36.998776,31.368956,302,6.577698,1
1,36.881995,31.386922,134,3.956093,1
2,36.989793,31.342006,333,2.475168,1
3,36.917928,31.054546,11,0.147884,1
4,36.846062,31.539636,67,2.416106,1


,Enlem,Boylam,Yukseklik,Egim,YANGIN_DURUMU
206,36.980810,29.922668,1476,0.446865,0
207,36.729281,29.725039,1670,10.803011,0
208,37.295220,30.326910,823,0.093112,0
209,37.205389,31.512686,1843,10.389173,0
210,36.899961,29.940635,1786,8.519497,0


In [48]:
import pandas as pd
import ee

# 1. TOPOGRAFYA (Önceki başarılı adımlarımız - Yükseklik ve Eğim)
elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Yukseklik')
slope_image = ee.Terrain.slope(elevation_image).rename('Egim')

# 2. YENİ ADIM: METEOROLOJİ (Sıcaklık)
# ERA5 Land veri setinden 2021 yaz dönemi (Temmuz-Ağustos) ortalamasını çekiyoruz
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
         .filterDate('2021-07-01', '2021-08-31') \
         .mean()

# Gelen veri Kelvin olduğu için 273.15 çıkararak Celsius'a çeviriyoruz
temp_c_image = era5.select('temperature_2m').subtract(273.15).rename('Sicaklik_C')

# 3. Tüm Verileri Tek Haritada Birleştirme (Yükseklik + Eğim + Sıcaklık)
combined_image = ee.Image.cat([elevation_image, slope_image, temp_c_image])

# 4. Koordinatlara Verileri Zımbalama
points_with_data = combined_image.sampleRegions(
    collection=combined_points, # 0 ve 1'lerimizin olduğu temel koordinatlar
    scale=1000,
    geometries=True
)

# 5. Python Tablosuna Çevirme İşlemi
def extract_all_features(feature):
    coords = feature.geometry().coordinates()
    return ee.Feature(None, {
        'Boylam': coords.get(0),
        'Enlem': coords.get(1),
        'Yukseklik': feature.get('Yukseklik'),
        'Egim': feature.get('Egim'),
        'Sicaklik_C': feature.get('Sicaklik_C'), # Yeni eklenen Sıcaklık sütunu
        'YANGIN_DURUMU': feature.get('YANGIN_DURUMU')
    })

final_dataset_step5 = points_with_data.map(extract_all_features)

#Veriyi Çekip Pandas Tablosu Yapma (Sütun listesini 6'ya çıkardık)
columns_to_fetch = ['Enlem', 'Boylam', 'Yukseklik', 'Egim', 'Sicaklik_C', 'YANGIN_DURUMU']

data_list_step5 = final_dataset_step5.reduceColumns(
    ee.Reducer.toList(len(columns_to_fetch)), columns_to_fetch
).get('list').getInfo()

# Eksik verileri (.dropna) temizleyerek tabloyu oluşturuyoruz
df_temp = pd.DataFrame(data_list_step5, columns=columns_to_fetch).dropna()

display(df_temp.head())
display(df_temp.tail())


,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,YANGIN_DURUMU
0,36.998776,31.368956,302,6.577698,27.957246,1
1,36.881995,31.386922,134,3.956093,29.786247,1
2,36.989793,31.342006,333,2.475168,29.828557,1
3,36.917928,31.054546,11,0.147884,30.978833,1
4,36.846062,31.539636,67,2.416106,29.078005,1


,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,YANGIN_DURUMU
198,36.980810,29.922668,1476,0.446865,21.663661,0
199,36.729281,29.725039,1670,10.803011,22.546458,0
200,37.295220,30.326910,823,0.093112,25.185609,0
201,37.205389,31.512686,1843,10.389173,21.070538,0
202,36.899961,29.940635,1786,8.519497,22.301253,0


In [49]:
import pandas as pd
import ee

#ÖNCEKİ BAŞARILI ADIMLAR (Yükseklik, Eğim ve Sıcaklık)
elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Yukseklik')
slope_image = ee.Terrain.slope(elevation_image).rename('Egim')

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
         .filterDate('2021-07-01', '2021-08-31') \
         .mean()
temp_c_image = era5.select('temperature_2m').subtract(273.15).rename('Sicaklik_C')

#RÜZGAR HIZI (u ve v vektörlerinden skaler hız hesabı)
u_wind = era5.select('u_component_of_wind_10m')
v_wind = era5.select('v_component_of_wind_10m')
wind_speed_image = u_wind.pow(2).add(v_wind.pow(2)).sqrt().rename('Ruzgar_Hizi')

# 3. Tüm Verileri Tek Haritada Birleştirme (Yükseklik + Eğim + Sıcaklık + Rüzgar Hızı)
combined_image = ee.Image.cat([elevation_image, slope_image, temp_c_image, wind_speed_image])

# 4. Koordinatlara Verileri Zımbalama
points_with_data = combined_image.sampleRegions(
    collection=combined_points,
    scale=1000,
    geometries=True
)

# 5. Python Tablosuna Çevirme İşlemi
def extract_all_features(feature):
    coords = feature.geometry().coordinates()
    return ee.Feature(None, {
        'Boylam': coords.get(0),
        'Enlem': coords.get(1),
        'Yukseklik': feature.get('Yukseklik'),
        'Egim': feature.get('Egim'),
        'Sicaklik_C': feature.get('Sicaklik_C'),
        'Ruzgar_Hizi': feature.get('Ruzgar_Hizi'), # Yeni eklenen Rüzgar sütunu
        'YANGIN_DURUMU': feature.get('YANGIN_DURUMU')
    })

final_dataset_step6 = points_with_data.map(extract_all_features)

# 6. Veriyi Çekip Pandas Tablosu Yapma (Sütun listesini 7'ye çıkardık)
columns_to_fetch = ['Enlem', 'Boylam', 'Yukseklik', 'Egim', 'Sicaklik_C', 'Ruzgar_Hizi', 'YANGIN_DURUMU']

data_list_step6 = final_dataset_step6.reduceColumns(
    ee.Reducer.toList(len(columns_to_fetch)), columns_to_fetch
).get('list').getInfo()

# Eksik verileri temizle
df_wind = pd.DataFrame(data_list_step6, columns=columns_to_fetch).dropna()


display(df_wind.head())
display(df_wind.tail())

,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,Ruzgar_Hizi,YANGIN_DURUMU
0,36.998776,31.368956,302,6.577698,27.957246,0.691825,1
1,36.881995,31.386922,134,3.956093,29.786247,0.440963,1
2,36.989793,31.342006,333,2.475168,29.828557,0.631214,1
3,36.917928,31.054546,11,0.147884,30.978833,0.607256,1
4,36.846062,31.539636,67,2.416106,29.078005,0.360979,1


,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,Ruzgar_Hizi,YANGIN_DURUMU
198,36.980810,29.922668,1476,0.446865,21.663661,0.892307,0
199,36.729281,29.725039,1670,10.803011,22.546458,0.480758,0
200,37.295220,30.326910,823,0.093112,25.185609,1.007540,0
201,37.205389,31.512686,1843,10.389173,21.070538,1.141420,0
202,36.899961,29.940635,1786,8.519497,22.301253,0.774359,0


In [50]:
import pandas as pd
import ee

#(Topografya ve Meteoroloji)
elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Yukseklik')
slope_image = ee.Terrain.slope(elevation_image).rename('Egim')

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
         .filterDate('2021-07-01', '2021-08-31') \
         .mean()
temp_c_image = era5.select('temperature_2m').subtract(273.15).rename('Sicaklik_C')

u_wind = era5.select('u_component_of_wind_10m')
v_wind = era5.select('v_component_of_wind_10m')
wind_speed_image = u_wind.pow(2).add(v_wind.pow(2)).sqrt().rename('Ruzgar_Hizi')

# BİTKİ ÖRTÜSÜ / YAKIT (MODIS NDVI)
# MOD13A2 veri setinden 2021 yaz dönemi ortalamasını çekip 0.0001 Scale Factor uyguluyoruz
ndvi_image = ee.ImageCollection("MODIS/061/MOD13A2") \
               .filterDate('2021-07-01', '2021-08-31') \
               .select('NDVI') \
               .mean() \
               .multiply(0.0001).rename('NDVI')

#Tüm Verileri Tek Haritada Birleştirme (Yükseklik + Eğim + Sıcaklık + Rüzgar + NDVI)
combined_image = ee.Image.cat([
    elevation_image, slope_image, temp_c_image, wind_speed_image, ndvi_image
])

#Koordinatlara Verileri Zımbalama
points_with_data = combined_image.sampleRegions(
    collection=combined_points,
    scale=1000,
    geometries=True
)

# 5. Python Tablosuna Çevirme İşlemi
def extract_all_features(feature):
    coords = feature.geometry().coordinates()
    return ee.Feature(None, {
        'Boylam': coords.get(0),
        'Enlem': coords.get(1),
        'Yukseklik': feature.get('Yukseklik'),
        'Egim': feature.get('Egim'),
        'Sicaklik_C': feature.get('Sicaklik_C'),
        'Ruzgar_Hizi': feature.get('Ruzgar_Hizi'),
        'NDVI': feature.get('NDVI'), # Yeni eklenen NDVI sütunu
        'YANGIN_DURUMU': feature.get('YANGIN_DURUMU')
    })

final_dataset_step7 = points_with_data.map(extract_all_features)


columns_to_fetch = ['Enlem', 'Boylam', 'Yukseklik', 'Egim', 'Sicaklik_C', 'Ruzgar_Hizi', 'NDVI', 'YANGIN_DURUMU']

data_list_step7 = final_dataset_step7.reduceColumns(
    ee.Reducer.toList(len(columns_to_fetch)), columns_to_fetch
).get('list').getInfo()

# Eksik verileri temizle
df_final = pd.DataFrame(data_list_step7, columns=columns_to_fetch).dropna()

display(df_final.head())
display(df_final.tail())

,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,Ruzgar_Hizi,NDVI,YANGIN_DURUMU
0,36.998776,31.368956,302,6.577698,27.957246,0.691825,0.375050,1
1,36.881995,31.386922,134,3.956093,29.786247,0.440963,0.396200,1
2,36.989793,31.342006,333,2.475168,29.828557,0.631214,0.390475,1
3,36.917928,31.054546,11,0.147884,30.978833,0.607256,0.441300,1
4,36.846062,31.539636,67,2.416106,29.078005,0.360979,0.292475,1


,Enlem,Boylam,Yukseklik,Egim,Sicaklik_C,Ruzgar_Hizi,NDVI,YANGIN_DURUMU
198,36.980810,29.922668,1476,0.446865,21.663661,0.892307,0.240950,0
199,36.729281,29.725039,1670,10.803011,22.546458,0.480758,0.225200,0
200,37.295220,30.326910,823,0.093112,25.185609,1.007540,0.243800,0
201,37.205389,31.512686,1843,10.389173,21.070538,1.141420,0.327375,0
202,36.899961,29.940635,1786,8.519497,22.301253,0.774359,0.229325,0
